In [1]:
from pamda import pamda
import statistics
from pprint import pp

In [ ]:
data1 = pamda.read_csv('../../outputs/geo_time_comparison_tests.csv')
data2 = pamda.read_csv('../../outputs/osrm_time_tests.csv')
# Merge data based on city1 and city2
data_map = {(row['city1'], row['city2']): row for row in data1}
for row in data2:
    key = (row['city1'], row['city2'])
    if key in data_map:
        data_map[key].update(row)
data = list(data_map.values())

FileNotFoundError: [Errno 2] No such file or directory: 'outputs/geo_time_comparison_tests.csv'

In [ ]:
columns = list(data[0].keys())
for i in columns:
    print(f"Column: {i}")

In [ ]:
time_columns = [i for i in columns if i.endswith('_time_ms') and 'osrm_' not in i or i in ['test_osrm_ch_time_ms', 'test_osrm_mld_time_ms']]
length_columns = [i for i in columns if i.endswith('_length_km')]

In [ ]:
avg_times = {i: sum(item[i] for item in data) / len(data) for i in time_columns}
std_times = {i: statistics.stdev(item[i] for item in data) for i in time_columns if len(data) > 1}

print("Average Times:")
pp(avg_times)
print("Standard Deviation of Times:")
pp(std_times)

In [ ]:
distance_errors = {i: [item[i] - item['test_google_length_km'] for item in data] for i in length_columns}
avg_distance_errors = {i: sum(distance_errors[i]) / len(distance_errors[i]) for i in length_columns}
print("Average Distance Errors:")
pp(avg_distance_errors)
avg_length = {i: sum(item[i] for item in data) / len(data) for i in length_columns}
print("Average Route Lengths:")
pp(avg_length)
avg_pct_error = {i: avg_distance_errors[i] / avg_length[i] * 100 for i in length_columns}
print("Average Percentage Error:")
pp(avg_pct_error)

In [ ]:
mape = {i: [abs(item[i] - item['test_google_length_km']) / item['test_google_length_km'] for item in data] for i in length_columns}
print("Mean Absolute Percentage Error (MAPE):")
pp({k:sum(v)/len(v) for k, v in mape.items()})

In [ ]:
def calculate_r_squared(y_true, y_pred):
    """
    Calculates the R-squared (coefficient of determination).

    Args:
        y_true (list): The true target values.
        y_pred (list): The predicted target values from the model.

    Returns:
        float: The R-squared value.
    """
    if len(y_true) != len(y_pred):
        raise ValueError("y_true and y_pred must have the same length.")

    # Calculate the mean of the true values
    y_true_mean = sum(y_true) / len(y_true)

    # Calculate the total sum of squares (SST)
    sst = sum([(y - y_true_mean)**2 for y in y_true])

    # Calculate the residual sum of squares (SSR)
    ssr = sum([(y_true[i] - y_pred[i])**2 for i in range(len(y_true))])

    # Calculate R-squared
    if sst == 0:
        # Handle the case where there is no variance in y_true
        return 1.0 if ssr == 0 else float('-inf')
    else:
        return 1 - (ssr / sst)

In [ ]:
r_sq = {i: calculate_r_squared([item['test_google_length_km'] for item in data], [item[i] for item in data]) for i in length_columns}
pp(r_sq)